In [18]:
# Imports
import yfinance as yf
import pandas as pd
import requests

**Question** 1: S&P 500 Additions

In [19]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
response = requests.get(url, headers=headers)

In [20]:
# 1. Create a DataFrame
df= pd.read_html(response.text)[0]
df.head()

/tmp/ipykernel_2019/2505045048.py:2: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df= pd.read_html(response.text)[0]


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [21]:
# 2.Extract the year
df['Date added'] = pd.to_datetime(df['Date added'], errors='coerce')
df['Year added'] = df['Date added'].dt.year

In [22]:
# Calculate the number of stocks added each year from 2020 onward
recent_additions = df[df['Year added'] >= 2020]
additions_per_year = recent_additions['Year added'].value_counts()

In [23]:
# 3. Year with the highest additions
highest_year = additions_per_year.idxmax()
print(f"Year with highest additions (since 2020): {int(highest_year)} ({additions_per_year.max()} additions)")

Year with highest additions (since 2020): 2025 (18 additions)


In [24]:
# Additional: Stocks in the index for more than 20 years (Added before 2006)
over_20_years = df[df['Year added'] < 2006].shape[0]
print(f"Current stocks in the index for over 20 years: {over_20_years}")

Current stocks in the index for over 20 years: 218


**Question** 2: Indexes YTD (as of 21 August 2026)

In [28]:
# Map friendly names to yfinance tickers
tickers_dict = {
    'US': '^GSPC', 'China': '000001.SS', 'Hong Kong': '^HSI',
    'Australia': '^AXJO', 'India': '^NSEI', 'Canada': '^GSPTSE',
    'Germany': '^GDAXI', 'UK': '^FTSE', 'Japan': '^N225',
    'Mexico': '^MXX', 'Brazil': '^BVSP'
}

# Download daily data for the required timeframe
data = yf.download(list(tickers_dict.values()), start='2026-01-01', end='2026-08-21',auto_adjust=False)['Close']

# Calculate YTD returns
returns = data.apply(lambda col: (col.dropna().iloc[-1] / col.dropna().iloc[0]) - 1)

# Rename the index back to country names for readability
returns.index = returns.index.map({v: k for k, v in tickers_dict.items()})

# Extract US return and compare
us_return = returns['US']
better_than_us = returns[returns > us_return]

print(f"S&P 500 YTD Return: {us_return * 100:.2f}%")
print(f"Indexes outperforming the US: {len(better_than_us)} out of {len(returns) - 1}")
if not better_than_us.empty:
    print(better_than_us.sort_values(ascending=False) * 100)

[*********************100%***********************]  11 of 11 completed

S&P 500 YTD Return: 11.41%
Indexes outperforming the US: 2 out of 10
Ticker
Japan     27.750745
Canada    14.057466
dtype: float64


**Question** 3: S&P 500 Market Corrections Analysis

In [32]:

# 1. Download S&P 500 historical data
sp500_data = yf.download('^GSPC', start='1950-01-01', auto_adjust=False)['Close']
sp500 = sp500_data.squeeze()
# 2. Identify all-time highs
rolling_max = sp500.cummax()
is_ath = sp500 == rolling_max
ath_dates = sp500[is_ath].index

corrections = []

# 3. Iterate through consecutive ATHs
for i in range(len(ath_dates) - 1):
    start_date = ath_dates[i]
    end_date = ath_dates[i+1]

    window = sp500.loc[start_date:end_date]
    if len(window) > 1:

        min_price = window.min()
        trough_date = window.idxmin()
        high_price = window.iloc[0] # The ATH price at start_date


        min_price = float(min_price)
        high_price = float(high_price)

        # 4. Calculate drawdown percentage
        drawdown = (high_price - min_price) / high_price * 100

        # 5. Filter for >= 5% drawdown
        if drawdown >= 5.0:
            # 6. Calculate duration from ATH to trough
            duration = (trough_date - start_date).days
            corrections.append({'Drawdown (%)': drawdown, 'Duration (Days)': duration})

df_corrections = pd.DataFrame(corrections)

# 7. Determine 25th, 50th (median), and 75th percentiles
if not df_corrections.empty:
    percentiles = df_corrections.quantile([0.25, 0.50, 0.75])
    print("Correction Percentiles (Drawdown and Duration):")
    print(percentiles)
else:
    print("No corrections found.")

[*********************100%***********************]  1 of 1 completed


Correction Percentiles (Drawdown and Duration):
      Drawdown (%)  Duration (Days)
0.25      6.234677            22.00
0.50      7.986358            40.50
0.75     14.019826            86.25


**Question** 4: Earnings Surprise Analysis for Amazon (AMZN)

In [34]:


# 1. Load earnings data
ticker_obj = yf.Ticker('AMZN')
earnings = ticker_obj.get_earnings_dates()

# Clean data: drop future earnings with missing Reported EPS
earnings = earnings.dropna(subset=['Reported EPS'])

# 2. Download historical price data
amzn_data = yf.download('AMZN',
                        start=earnings.index.min() - pd.Timedelta(days=10),
                        end=earnings.index.max() + pd.Timedelta(days=10),
                        auto_adjust=False)['Close']
amzn_prices = amzn_data.squeeze()
results = []

# 3. Calculate 2-day percentage changes
for date, row in earnings.iterrows():
    # Convert timezone-aware datetime to naive for index matching
    e_date = date.tz_localize(None).normalize()

    try:
        # Find the index of the closest trading day (Day 2)
        day2_idx = amzn_prices.index.get_indexer([e_date], method='bfill')[0]

        if 0 < day2_idx < len(amzn_prices) - 1:

            day1_price = float(amzn_prices.iloc[day2_idx - 1])
            day3_price = float(amzn_prices.iloc[day2_idx + 1])

            # Return = Close_Day3 / Close_Day1 - 1
            ret_2d = (day3_price / day1_price) - 1


            surprise_val = row.get('Surprise(%)', 0)
            if pd.isna(surprise_val):
                surprise_val = 0.0

            results.append({
                'Surprise': float(surprise_val),
                '2_Day_Return': ret_2d
            })
    except Exception as e:
        continue

df_results = pd.DataFrame(results)

# 4. Filter for positive surprises and calculate metrics
if not df_results.empty:
    positive_surprises = df_results[df_results['Surprise'] > 0]

    if not positive_surprises.empty:
        median_return = positive_surprises['2_Day_Return'].median()
        correlation = df_results['2_Day_Return'].corr(df_results['Surprise'])

        print(f"Median 2-day return for positive surprises: {median_return * 100:.2f}%")
        print(f"Correlation (Surprise vs. Return): {correlation:.4f}")
    else:
        print("No positive surprise events found.")
else:
    print("Could not calculate returns. Check data availability.")

[*********************100%***********************]  1 of 1 completed

Median 2-day return for positive surprises: 0.35%
Correlation (Surprise vs. Return): 0.2191


**Question** 5: Brainstorm potential idea for your capstone project



I want to build a predictive classification model focusing on the US mid-cap technology sector to forecast whether a stock will outperform the Russell 2000 index over a 60-day investment horizon. Instead of purely relying on technical indicators like RSI or moving averages, I want to incorporate alternative data via NLP sentiment analysis. By processing the management discussion sections of quarterly SEC 10-Q filings and cross-referencing them with insider trading volumes, I hope to identify companies that are signaling quiet confidence before the broader market catches on.

**Question** 6: Investigate new metrics

To support my capstone project, I would pull three additional time series:

CBOE Volatility Index (VIX): Retrieved via yf.download('^VIX'). This metric gauges broader market fear. I believe high VIX environments might mute the positive price impact of strong earnings reports, serving as a vital macro-filter for my strategy.

US Treasury Yield Curve (10-Year minus 2-Year): Accessed using the pandas_datareader library linked to the FRED (Federal Reserve Economic Data) API (T10Y2Y). The slope of the yield curve is a reliable proxy for economic expansion or contraction, helping contextualize whether the market favors growth or value stocks at any given time.

Insider Net Buying Volume: While trickier to scrape natively through yfinance, this can be retrieved via the SEC EDGAR database (Form 4 filings). Tracking the ratio of insider buys to sells within a specific mid-cap company provides a strong signal of internal conviction that technical indicators cannot capture.